In [0]:
dbutils.fs.mkdirs("/Volumes/misha_azure/default/global_sales/bronze")
dbutils.fs.mkdirs("/Volumes/misha_azure/default/global_sales/silver")
dbutils.fs.mkdirs("/Volumes/misha_azure/default/global_sales/gold")

True

In [0]:
BRONZE_BASE_PATH = "/Volumes/misha_azure/default/global_sales/bronze"

In [0]:
#import functions
from pyspark.sql.functions import current_timestamp, col

In [0]:
def ingest_to_bronze(file_name, table_name):
    try:
        print(f"Starting Bronze ingestion for table: {table_name}")

        file_path = f"{BRONZE_BASE_PATH}/raw_files/{file_name}"
        print(f"Reading file from path: {file_path}")

        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(file_path)

        print(f"Raw record count for {table_name}: {df.count()}")

        df_bronze = df \
            .withColumn("ingestion_ts", current_timestamp()) \
            .withColumn("source_file", col("_metadata.file_path"))

        print(f"Writing Bronze table to path: {BRONZE_BASE_PATH}/{table_name}")

        df_bronze.write.format("delta") \
            .mode("overwrite") \
            .save(f"{BRONZE_BASE_PATH}/{table_name}")

        print(f"Bronze ingestion completed successfully for: {table_name}")

    except Exception as e:
        print(f"ERROR during Bronze ingestion for {table_name}")
        print("Error details:", str(e))
        raise

In [0]:
#Ingestions
ingest_to_bronze(
    file_name="sales_transactions.csv",
    table_name="sales_transactions"
)

Starting Bronze ingestion for table: sales_transactions
Reading file from path: /Volumes/misha_azure/default/global_sales/bronze/raw_files/sales_transactions.csv
Raw record count for sales_transactions: 10150
Writing Bronze table to path: /Volumes/misha_azure/default/global_sales/bronze/sales_transactions
Bronze ingestion completed successfully for: sales_transactions


In [0]:
ingest_to_bronze(
    file_name="products.csv",
    table_name="products"
)

Starting Bronze ingestion for table: products
Reading file from path: /Volumes/misha_azure/default/global_sales/bronze/raw_files/products.csv
Raw record count for products: 306
Writing Bronze table to path: /Volumes/misha_azure/default/global_sales/bronze/products
Bronze ingestion completed successfully for: products


In [0]:
ingest_to_bronze(
    file_name="customers.csv",
    table_name="customers"
)

Starting Bronze ingestion for table: customers
Reading file from path: /Volumes/misha_azure/default/global_sales/bronze/raw_files/customers.csv
Raw record count for customers: 2040
Writing Bronze table to path: /Volumes/misha_azure/default/global_sales/bronze/customers
Bronze ingestion completed successfully for: customers


In [0]:
ingest_to_bronze(
    file_name="stores.csv",
    table_name="stores"
)

Starting Bronze ingestion for table: stores
Reading file from path: /Volumes/misha_azure/default/global_sales/bronze/raw_files/stores.csv
Raw record count for stores: 50
Writing Bronze table to path: /Volumes/misha_azure/default/global_sales/bronze/stores
Bronze ingestion completed successfully for: stores


In [0]:
ingest_to_bronze(
    file_name="returns.csv",
    table_name="returns"
)

Starting Bronze ingestion for table: returns
Reading file from path: /Volumes/misha_azure/default/global_sales/bronze/raw_files/returns.csv
Raw record count for returns: 800
Writing Bronze table to path: /Volumes/misha_azure/default/global_sales/bronze/returns
Bronze ingestion completed successfully for: returns


In [0]:
ingest_to_bronze(
    file_name="sales_targets.csv",
    table_name="sales_targets"
)

Starting Bronze ingestion for table: sales_targets
Reading file from path: /Volumes/misha_azure/default/global_sales/bronze/raw_files/sales_targets.csv
Raw record count for sales_targets: 48
Writing Bronze table to path: /Volumes/misha_azure/default/global_sales/bronze/sales_targets
Bronze ingestion completed successfully for: sales_targets


In [0]:
#Validate Bronze Tables
spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/sales_transactions") \
    .count()

2026-01-03 07:57:38,616 | INFO | Received command c on object id p0


10150

In [0]:
spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/products") \
    .count()

306

In [0]:
spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/customers") \
    .count()

2040

In [0]:
spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/stores") \
    .count()

2026-01-03 07:57:46,795 | INFO | Received command c on object id p0


50

In [0]:
#Schema check
spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/sales_transactions") \
    .printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/products") \
    .printSchema()


root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- list_price: integer (nullable = true)
 |-- active_flag: boolean (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/customers") \
    .printSchema()


root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/stores") \
    .printSchema()


root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- region: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- open_date: date (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

